# 六足ロボットの円柱押し：学生用APIを一歩ずつ学ぶ

**報酬・観測・行動のどこを変えると、学習結果がどう変わるか**を調べる教材です。
18関節の六足ロボット、固定した歩行モデル、MuJoCoの接触物理を使い、上位の動作をTorchRL PPOで学習します。

1. [Colabでデモを開く](https://colab.research.google.com/github/hashimoto-robotics-lab/hexapod-transport-rl/blob/main/notebooks/student_api_colab.ipynb)をクリックします。
2. 「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選びます。CPUでも短いデモは動きます。
3. 最初のセルでGitHubから教材をcloneし、依存ライブラリを準備します。
4. 上から順番に実行します。初回のインストール・GPUカーネルのコンパイルには数分かかる場合があります。

標準課題：円柱の周囲でロボットの初期位置・向きがランダム、目標方向・距離もランダム。
成功は**位置誤差5 cm未満・円柱速度0.06 m/s未満を0.4秒保持**です。円柱の向きは採点しません。
このノートブックは初期設定では短い**操作練習**です。本学習への切り替えはStep 1で行います。

## Step 0 — 準備する

GitHubから教材を取得し、実行に必要なライブラリを準備します。
取得先のブランチは`REVISION`で指定します。再実行時は取得済みの教材を使います。

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = "https://github.com/hashimoto-robotics-lab/hexapod-transport-rl.git"
REVISION = "main"
PROJECT_DIR = Path("/content/hexapod_transport_rl")

if not PROJECT_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REVISION,
        REPOSITORY, str(PROJECT_DIR),
    ], check=True)

sys.path.insert(0, str(PROJECT_DIR / "tools"))
from colab_setup import prepare_colab

prepare_colab(PROJECT_DIR)

## Step 1 — 学習時間と保存先を決める

`PROFILE = "demo"`は、2秒のエピソードでAPIの操作を練習します。
`"research"`に変えると、40秒の課題を655,360サンプルで学習します。
本学習ではT4 GPUを選びます。短いデモの成功率は学習性能の評価には使いません。
保存先には時刻を付けるので、セルを再実行して実験結果を残せます。

In [ ]:
from dataclasses import replace
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Video, display

from hexapod_transport_rl import (
    ContinuousVelocity, CylinderPushTask, DiscreteDirections, ObservationTerm,
    PPOSettings, compare, evaluate, load_task, load_task_file, record_video, train,
)

PROFILE = "demo"  # "research" にすると本学習
base = CylinderPushTask()

if PROFILE == "demo":
    base = replace(base, config=replace(base.config, episode_seconds=2.0))
    options = dict(steps=32, num_envs=2, horizon=8, validation_envs=2,
                   validate_every=1, settings=PPOSettings(epochs=1, minibatch_size=32))
    evaluation_episodes, comparison_seeds = 4, (42, 43)
else:
    options = dict(steps=655_360, num_envs=256, horizon=64,
                   validation_envs=256, validate_every=10)
    evaluation_episodes, comparison_seeds = 256, (42, 43, 44)

WORK_DIR = PROJECT_DIR / "runs" / datetime.now().strftime("tutorial_%Y%m%d_%H%M%S_%f")
WORK_DIR.mkdir(parents=True)
print("保存先:", WORK_DIR)

## Step 2 — `reset`・`step`・観測・報酬を見てみる

環境は`task.make_env()`で作り、`reset()`で開始、`step()`で0.2秒進めます。
`num_envs`は独立した世界の数で、各世界に六足ロボットは1台ずつ存在します。
`observation`が方策に渡る入力です。標準は14成分で、行動は停止＋世界座標の8方向の9択です。
`env.state`では、報酬を設計するときに使える名前付き物理量を確認できます。

In [ ]:
env = base.make_env(num_envs=1, backend="cpu", render_mode="rgb_array")
env.set_seed(42)
observation = env.reset()
print("方策に渡す観測:", observation["observation"][0].tolist())
print("目標位置 [m]:", env.state.goal_position[0].tolist())

plt.figure(figsize=(6, 4))
plt.imshow(env.render())
plt.axis("off")
plt.show()

observation["action"] = torch.tensor([1])  # 0=停止、1=東、2=北東、3=北、…
next_observation = env.step(observation)["next"]
print("報酬:", next_observation["reward"].item())
print("内訳:", {name: value.item() for name, value in next_observation["reward_terms"].items()})
env.close()

## Step 3 — 標準設定で学習する

`train`に課題・保存先・学習条件を渡します。
上位の方策をランダムな重みから学習し、歩行モデルは固定します。
`run.checkpoint`は検証で選んだ方策、`run.latest`は最後に更新した方策です。

In [ ]:
run = train(base, WORK_DIR / datetime.now().strftime("baseline_%H%M%S_%f"), seed=42, **options)

history = pd.read_json(run.directory / "progress.json")
display(history[["steps", "reward", "mean_distance_m", "successes"]])

## Step 4 — 評価する・動画を見る

学習に使わないseedで成功率と最終距離を測ります。
動画はMuJoCoでの実際の動きを保存したものです。

In [ ]:
report = run.evaluate(episodes=evaluation_episodes, seed=240000)
print(f"成功: {report['successes']} / {report['episode_count']}")
print(f"平均の最終距離: {report['mean_distance_m']:.3f} m")

run.video(WORK_DIR / "baseline.mp4", seed=240000)
display(Video(str(WORK_DIR / "baseline.mp4"), embed=True, width=640))

## Step 5 — 報酬の重みを変えて、もう一度学習する

円柱を目標へ近づけたときの報酬`goal_progress`を、**20から40**に変更します。
`replace`で標準課題をコピーし、報酬だけ変更します。
同じ学習seedとサンプル数で新しい方策を学習し、共通の評価seedで比較します。

**試してみよう：** `40.0`を`10.0`や`80.0`に変えて実行し、成功率や最終距離を比べましょう。
報酬の大きさが異なるため、性能の比較には成功率と最終距離を使います。

In [ ]:
reward_task = replace(base, rewards={**base.rewards, "goal_progress": 40.0})
reward_run = train(reward_task, WORK_DIR / datetime.now().strftime("reward_weights_%H%M%S_%f"), seed=42, **options)
reward_report = reward_run.evaluate(episodes=evaluation_episodes, seed=240000)

display(pd.DataFrame([
    {"条件": name,
     "成功率": result["successes"] / result["episode_count"],
     "最終距離 [m]": result["mean_distance_m"]}
    for name, result in [("標準の報酬", report), ("進捗の重みを40に変更", reward_report)]
]))

reward_run.video(WORK_DIR / "reward_weights.mp4", seed=240000)
display(Video(str(WORK_DIR / "reward_weights.mp4"), embed=True, width=640))

## Step 6 — 観測・行動を変える

観測は、円柱からロボット・目標へのXYとロボットの向きだけにすると6成分になります。
`scale`は観測を割る値です。ここでは標準観測と同じくXYを直径0.5 mで割ります。
行動は`DiscreteDirections(4)`で停止＋東西南北の5択、`ContinuousVelocity()`で連続XYになります。
これらの課題も`train`へ渡せます。Step 8では標準課題と比較します。

In [ ]:
observation_task = replace(base, observations=(
    ObservationTerm("cargo_to_robot", scale=0.5),
    ObservationTerm("cargo_to_goal", scale=0.5),
    "robot_heading",
))
action_task = replace(base, actions=DiscreteDirections(count=4))
continuous_task = replace(base, actions=ContinuousVelocity())

print("停止＋4方向の選択肢:", action_task.actions.directions)

## Step 7 — 自分で報酬関数を書いて学習する

次は、**目標の10 cm以内で円柱が速く動くほど減点する**関数を追加します。
`before`と`after`から前後の状態を参照でき、並列の世界ごとに1つの数値を返します。
`weight=-2.0`は減点の強さ、`per_second=True`は時間に応じて積分する指定です。
観測・行動は標準課題と同じです。

`%%writefile`は、関数と課題を小さなPythonファイルに保存します。
このファイルを`load_task_file`で読み込むと、学習した重みと一緒に報酬の定義も保存できます。

**試してみよう：** `0.10`を`0.20`に変えると、目標の20 cm手前から減速を促す報酬になります。

In [ ]:
%%writefile {WORK_DIR}/my_reward.py
from dataclasses import replace
from hexapod_transport_rl import CylinderPushTask, RewardTerm

def near_goal_speed(before, after):
    return after.cargo_speed.square() * (after.goal_distance < 0.10)

baseline = CylinderPushTask()
TASK = replace(baseline, rewards={
    **baseline.rewards,
    "near_goal_speed": RewardTerm(near_goal_speed, weight=-2.0, per_second=True),
})

In [ ]:
custom_task = load_task_file(WORK_DIR / "my_reward.py")
custom_task = replace(custom_task, config=base.config)
custom_run = train(custom_task, WORK_DIR / datetime.now().strftime("custom_reward_%H%M%S_%f"), seed=42, **options)
custom_report = custom_run.evaluate(episodes=evaluation_episodes, seed=240000)

print(f"成功: {custom_report['successes']} / {custom_report['episode_count']}")
print(f"平均の最終距離: {custom_report['mean_distance_m']:.3f} m")
custom_run.video(WORK_DIR / "custom_reward.mp4", seed=240000)
display(Video(str(WORK_DIR / "custom_reward.mp4"), embed=True, width=640))

## Step 8 — 複数のseedで比較する

標準、報酬の重み、独自報酬、観測、行動の5条件を比較します。
各条件で同じサンプル数を使い、学習seedごとに新しい方策を学習します。
評価seedは共通です。`research`では各条件3つの学習seed・256回の評価を使います。

表の成功率と最終距離を確認し、変更した項目が結果にどう影響したか考えてみましょう。

In [ ]:
comparison_dir = WORK_DIR / datetime.now().strftime("comparison_%H%M%S_%f")
comparison = compare(
    {"baseline": base, "reward_weights": reward_task, "custom_reward": custom_task,
     "observation": observation_task, "action": action_task},
    comparison_dir, seeds=comparison_seeds,
    evaluation_episodes=evaluation_episodes, evaluation_seed=240000, **options,
)
display(pd.read_csv(comparison_dir / "comparison.csv"))
display(pd.DataFrame(comparison["summary"]).T)

## Step 9 — 既存の学習済み方策を再利用する

同梱の`cylinder_push.pt`は、これまでの実験で学習した参照方策です。
`load_task`でその重みと一致する観測・行動・前処理を復元し、40秒の標準課題を評価します。
この方策は**目標方向に合わせて45度刻みで座標を回す前処理**を使っています。
前処理を使わないStep 3の新規学習とは別条件なので、そのまま学習性能を比較しないでください。

続きから学習するときは、`train(..., initial_checkpoint=reference)`を指定できます。
互換性のある観測・行動のまま報酬を変更する用途で使います。

In [ ]:
reference = PROJECT_DIR / "checkpoints/cylinder_push.pt"
reference_task = load_task(reference)
print("既存方策の前処理:", reference_task.rotation)
print("評価の制限時間:", reference_task.config.episode_seconds, "秒")
reference_report = evaluate(reference, episodes=evaluation_episodes, seed=250000)
print(f"参照方策の成功: {reference_report['successes']} / {reference_report['episode_count']}")
reference_video = record_video(reference, WORK_DIR / "reference.mp4", seed=250000)
display(Video(reference_video["video"], embed=True, width=640))

# 続きから学習する場合（任意。新しい出力先を指定）:
# continued = train(reference_task, WORK_DIR / "continued", seed=46,
#                   initial_checkpoint=reference, **options)

## Step 10 — 結果を保存して、次の実験へ

`task.json`に設定、`task_sources/`に独自関数、`environment.json`に依存ライブラリの版が残ります。
`progress.json`は学習経過、`validations.json`は検証、`evaluation_*.json`は独立評価です。
Colabを終了する前に結果をダウンロードしてください。学習済み重みだけでなく、フォルダごと保存すると独自関数も持ち出せます。

次の課題例：距離の重みだけ変える／速度の観測を1つずつ外す／停止の有無や4・8・16方向を比較する。
仮説・変更した項目・seed・学習サンプル数・成功率・最終距離を記録すると、卒論の実験表につなげられます。

In [ ]:
import shutil

archive = shutil.make_archive(str(WORK_DIR), "zip", root_dir=WORK_DIR.parent, base_dir=WORK_DIR.name)
print("実験結果ZIP:", archive)

DOWNLOAD_RESULTS = False  # Colabで手元に保存するときは True に変更
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive)

詳しい項目一覧は`docs/student-api.md`、Pythonの課題ファイルは`examples/student/`を参照してください。

Colabに入っているPyTorchを使い、MuJoCo・TorchRLなどの依存ライブラリを準備します。
検証環境はPython 3.12、CPU版PyTorch 2.14.1とCUDA版PyTorch 2.11.0です。